# Tome 6, Chapter 7: The Lebesgue Integral (Python companion)

Exercises first, solutions at the end. Integration on a computer: Riemann sums against Lebesgue sums and the layer-cake formula, Monte Carlo and quasi-Monte Carlo against grids in high dimension, the typewriter sequence, escaping mass and the failure of Monte Carlo on non-uniformly-integrable variables, and the Dirichlet integral with Feynman's trick. Requirements: numpy, scipy and matplotlib.

*Before each solution, predict what the output will look like.*

## Exercises

**P7.1 (★) Sorting the coins.** For $f(x)=x^2+\frac12\sin(12x)$ on $[0,1]$, compute Riemann sums (cut the variable into $n$ pieces) and Lebesgue sums $\sum_j y_j\,\lambda\{y_j\le f<y_{j+1}\}$ (cut the values into $n$ levels, estimating each level set's length on a fine grid). Then do it the way a computer would for data: sort $N$ sampled values of $f$ and read the integral off the sorted list (the layer-cake formula of Exercise 7.4). Which sum cares whether $f$ is continuous?

**P7.2 (★★) CS application: Monte Carlo against grids.** Estimate $\int_{[0,1]^d}\prod_{i=1}^d\frac\pi2\sin(\pi x_i)\,dx=1$ in dimensions $d=1,2,5,10,20$ with a budget of about $10^6$ evaluations, by (a) the midpoint rule on a product grid, (b) plain Monte Carlo, (c) a scrambled Sobol sequence (quasi-Monte Carlo). Explain the error of each: why does Monte Carlo not care about the dimension, and what does the strong law (Tome 3) have to do with the Lebesgue integral here?

**P7.3 (★) The typewriter.** For the functions $f_n$ of Exercise 7.2, compute $\int f_n$ and the sequence $(f_n(x))_n$ at $x=1/3$ and at a random $x$, for $n<2^{12}$. Find the indices where $f_n(x)=1$, and extract a subsequence that converges to $0$ at *every* point.

**P7.4 (★★) Escaping mass, seen by probability.** Let $U$ be uniform on $]0,1[$, $X_n=n\mathbf 1_{\{U<1/n\}}$ and $Y_n=\sqrt n\,\mathbf 1_{\{U<1/n\}}$. Estimate $\mathbb E[X_n]$, $\mathbb E[Y_n]$ and the tail quantities $\sup_n\mathbb E[|X_n|\mathbf 1_{\{|X_n|>M\}}]$ for several $M$, by simulation and exactly. Then run a Monte Carlo estimate of $\mathbb E[X_n]$ with $10^4$ samples for $n=10^6$: what goes wrong, and how is it related to uniform integrability?

**P7.5 (★★) The Dirichlet integral and Feynman's trick.** Plot $R\mapsto\int_0^R\frac{\sin x}x\,dx$ and check that it oscillates around $\frac\pi2$ with amplitude about $\frac1R$. Check $F(t)=\int_0^\infty e^{-tx}\frac{\sin x}x\,dx=\frac\pi2-\arctan t$ (Exercise 7.9) and $I(t)=\int_0^\infty e^{-x^2}\cos(tx)\,dx=\frac{\sqrt\pi}2e^{-t^2/4}$ (Exercise 7.7) with numerical quadrature, and compare $F'$ computed by differentiating under the integral with a finite difference.

## Solutions

### P7.1

In [ ]:
import numpy as np
f = lambda x: x ** 2 + 0.5 * np.sin(12 * x)
xs = np.linspace(0, 1, 2_000_001)[:-1] + 0.5 / 2_000_000      # fine grid used to measure level sets
fx = f(xs)
exact = 1 / 3 + 0.5 * (1 - np.cos(12)) / 12
print(f"exact integral {exact:.8f}")
lo, hi = fx.min(), fx.max()
for n in [10, 100, 1000]:
    x = (np.arange(n) + 0.5) / n
    riemann = f(x).mean()
    y = np.linspace(lo, hi, n + 1)
    counts = np.histogram(fx, bins=y)[0] / len(fx)          # lambda{y_j <= f < y_j+1}
    lebesgue = np.sum(y[:-1] * counts)                       # lower Lebesgue sum
    print(f"n={n:5d}: Riemann {riemann:.8f} (error {riemann-exact:+.1e})   lower Lebesgue {lebesgue:.8f} (error {lebesgue-exact:+.1e})")

rng = np.random.default_rng(0)
v = np.sort(f(rng.random(100_000)))
# layer cake: for nonnegative g, int g = int_0^inf P(g > s) ds; here shift by the minimum
s = v - v[0]
layer = v[0] + np.sum(np.diff(s) * (1 - np.arange(1, len(v)) / len(v)))
print(f"layer cake from sorted samples: {layer:.5f}   plain sample mean: {v.mean():.5f}")

The midpoint Riemann sums converge like $\frac1{n^2}$ for this smooth function; the lower Lebesgue sums converge like $\frac{\text{range}}n$, since each level loses at most one level width. So for a smooth function on an interval Riemann is the better *algorithm*; Lebesgue's sums are better *theory*, because they never look at where in $[0,1]$ the values are taken. The layer-cake computation shows the point: the sorted list of values (here, sample values) is all one needs, and it equals the sample mean exactly (summation by parts), whatever order the samples came in. A Riemann sum needs $f$ to vary little on small intervals; a Lebesgue sum needs only that level sets be measurable.

### P7.2

In [ ]:
import numpy as np
from scipy.stats import qmc
g1 = lambda x: np.pi / 2 * np.sin(np.pi * x)
def F(X): return np.prod(g1(X), axis=1)
budget = 2 ** 20
rng = np.random.default_rng(1)
for d in [1, 2, 5, 10, 20]:
    m = max(1, int(round(budget ** (1 / d))))                   # points per axis for the grid
    if m ** d <= 4 * budget and m > 1:
        grid1 = (np.arange(m) + 0.5) / m
        grid = g1(grid1).mean() ** d               # the product rule factorises over the axes
        grid_err = abs(grid - 1)
    else:
        grid_err = float("nan")
    errs_mc, errs_q = [], []
    for rep in range(5):
        errs_mc.append(abs(F(rng.random((budget, d))).mean() - 1))
        S = qmc.Sobol(d, scramble=True, seed=rep).random_base2(20)
        errs_q.append(abs(F(S).mean() - 1))
    sigma = np.sqrt((np.pi ** 2 / 8) ** d - 1)
    print(f"d={d:2d}: grid {m:7d}^d pts error {grid_err:.1e} | MC error {np.mean(errs_mc):.1e} (sigma/sqrt N = {sigma/np.sqrt(budget):.1e}) | Sobol {np.mean(errs_q):.1e}")

With $m$ points per axis, the grid costs $m^d$ evaluations, and the midpoint rule makes an error of order $m^{-2}=N^{-2/d}$ on each axis. In dimension $1$ or $2$ this is far better than anything random. In dimension $10$ the budget allows only $4$ points per axis, and in dimension $20$ only $2$: then the grid evaluates $\bigl(\frac\pi2\sin\frac\pi4\bigr)^{20}\approx8.2$ instead of $1$. This is the *curse of dimensionality*.

Monte Carlo averages $N$ independent copies of $F(X)$ with $X$ uniform on $[0,1]^d$. By the strong law of large numbers the average tends to $\mathbb E[F(X)]$, which *is* the Lebesgue integral $\int_{[0,1]^d}F\,d\lambda$; by the central limit theorem the error is about $\sigma/\sqrt N$ with $\sigma^2=\operatorname{Var}F(X)=(\frac{\pi^2}8)^d-1$, and the table confirms it. The rate $N^{-1/2}$ does not depend on $d$; only $\sigma$ does. No grid, no partition of the domain: the only thing used is the measure of sets, exactly Lebesgue's point of view. Quasi-Monte Carlo replaces random points by well spread deterministic ones (Sobol, 1967) and achieves errors close to $N^{-1}$ (up to logarithms) for smooth integrands of moderate effective dimension (in the table it wins by orders of magnitude up to $d=10$, and its advantage has almost vanished at $d=20$); this is how finance and computer graphics compute high-dimensional integrals.

### P7.3

In [ ]:
import numpy as np
def typewriter(n, x):
    k = n.bit_length() - 1; j = n - 2 ** k
    return 1 if j * 2.0 ** -k <= x <= (j + 1) * 2.0 ** -k else 0
for x in [1 / 3, np.random.default_rng(2).random()]:
    hits = [n for n in range(1, 2 ** 12) if typewriter(n, x)]
    print(f"x={x:.6f}: f_n(x)=1 for {len(hits)} indices n < 2^12, e.g. {hits[:8]} ...")
print("integrals 2^-k:", [2.0 ** -(n.bit_length() - 1) for n in [1, 2, 3, 4, 8, 16, 1000, 4000]])
sub = [2 ** k for k in range(12)]              # f_{2^k} = 1 on [0, 2^-k]
xs = np.random.default_rng(3).random(5)
print("subsequence f_{2^k} at random points:", [[typewriter(n, x) for n in sub] for x in xs][:2])

Each pass $k$ of the typewriter sweeps $[0,1]$ with intervals of length $2^{-k}$, so every $x$ is hit at least once per pass: $f_n(x)=1$ for at least $12$ indices below $2^{12}$, and infinitely often in all, while $f_n(x)=0$ infinitely often too. So $(f_n(x))$ converges nowhere, although $\int f_n=2^{-k}\to0$. The subsequence $f_{2^k}=\mathbf 1_{[0,2^{-k}]}$ tends to $0$ at every $x>0$, hence a.e. (at $x=0$ it is constantly $1$): this is Proposition 7.13(b) at work, which picks $n_k$ with $\lambda\{|f_{n_k}|>2^{-k}\}<2^{-k}$ and applies Borel--Cantelli. To get convergence at every point including $0$, take the last interval of each pass, $f_{2^{k+1}-1}=\mathbf 1_{[1-2^{-k},1]}$, and the point $1$ becomes the bad one: some point always is, unless one takes intervals avoiding it, such as $f_{2^k+1}=\mathbf 1_{[2^{-k},2^{1-k}]}$ for $k\ge1$, which tends to $0$ everywhere.

### P7.4

In [ ]:
import numpy as np
rng = np.random.default_rng(4)
U = rng.random(10 ** 6)
for n in [10, 100, 1000, 10 ** 4]:
    X = n * (U < 1 / n); Y = np.sqrt(n) * (U < 1 / n)
    print(f"n={n:6d}: E X_n ~ {X.mean():.3f} (exact 1)   E Y_n ~ {Y.mean():.4f} (exact {1/np.sqrt(n):.4f})")
for M in [10, 100, 1000]:
    # E[X_n 1{X_n > M}] = 1 if n > M, else 0; for Y_n it is 1/sqrt n if sqrt n > M
    print(f"M={M:5d}: sup_n E[X_n; X_n>M] = 1    sup_n E[Y_n; Y_n>M] = sup_(n>M^2) n^-1/2 < 1/M = {1/M}")
est = [(10 ** 6 * (rng.random(10 ** 4) < 1e-6)).mean() for _ in range(20)]
print("20 Monte Carlo estimates of E[X_n] = 1 with n = 10^6 and 10^4 samples:", [float(e) for e in est])

$\mathbb E[X_n]=1$ for every $n$, but $X_n\to0$ almost surely: the expectation is carried by an event of probability $\frac1n$ on which $X_n$ is huge. The family is not uniformly integrable, since $\mathbb E[X_n\mathbf 1_{\{X_n>M\}}]=1$ as soon as $n>M$, and Vitali's theorem (Theorem 7.16) says this is exactly why $\mathbb E[X_n]\not\to\mathbb E[\lim X_n]=0$. For $Y_n$ the tails are uniformly small, and $\mathbb E[Y_n]\to0$ as Vitali predicts.

The Monte Carlo experiment shows the practical side: with $10^4$ samples the event $\{U<10^{-6}\}$ almost never occurs, so the estimate is $0$ nearly every time, and very rarely $100$. The estimator is unbiased, but the variance $n-1$ is enormous, and a typical run is badly wrong. Heavy, non-uniformly-integrable tails are the classical failure mode of Monte Carlo; *importance sampling* fixes it by sampling more often where the mass hides.

### P7.5

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad
from scipy.special import sici
R = np.linspace(1, 60, 2000)
Si = sici(R)[0]                       # Si(R) = int_0^R sin x / x dx
plt.figure(figsize=(7, 3)); plt.plot(R, Si); plt.axhline(np.pi / 2, color="k", lw=0.7)
plt.plot(R, np.pi / 2 + 1 / R, "--", lw=0.7); plt.plot(R, np.pi / 2 - 1 / R, "--", lw=0.7); plt.title("Si(R) and pi/2 +- 1/R"); plt.show()
print("max over R in [30,60] of R |Si(R) - pi/2|:", np.max(R[R > 30] * np.abs(Si[R > 30] - np.pi / 2)))

for t in [0.1, 0.5, 1, 3]:
    Ft = quad(lambda x: np.exp(-t * x) * np.sinc(x / np.pi), 0, np.inf, limit=400)[0]
    It = quad(lambda x: np.exp(-x * x) * np.cos(t * x), 0, np.inf)[0]
    dF = -quad(lambda x: np.exp(-t * x) * np.sin(x), 0, np.inf, limit=400)[0]     # differentiate under the integral
    h = 1e-4
    fd = (quad(lambda x: np.exp(-(t + h) * x) * np.sinc(x / np.pi), 0, np.inf, limit=400)[0]
          - quad(lambda x: np.exp(-(t - h) * x) * np.sinc(x / np.pi), 0, np.inf, limit=400)[0]) / (2 * h)
    print(f"t={t}: F {Ft:.8f} vs pi/2-arctan t {np.pi/2-np.arctan(t):.8f} | I {It:.8f} vs {np.sqrt(np.pi)/2*np.exp(-t*t/4):.8f} | "
          f"F' under integral {dF:.6f}, finite difference {fd:.6f}, -1/(1+t^2) {-1/(1+t*t):.6f}")

$\operatorname{Si}(R)-\frac\pi2=-\int_R^\infty\frac{\sin x}x\,dx\approx-\frac{\cos R}R$ by the integration by parts of Exercise 7.9 (with $t=0$), so the partial integrals oscillate around $\frac\pi2$ with amplitude close to $\frac1R$, and $R\,|\operatorname{Si}(R)-\frac\pi2|$ stays just below $1$. The convergence is real but only *conditional*: $\int_0^\infty|\frac{\sin x}x|\,dx=\infty$, so $\frac{\sin x}x$ is not Lebesgue integrable on $[0,\infty[$, and the value $\frac\pi2$ is a limit of integrals, not an integral. The factor $e^{-tx}$ makes the integrand integrable for $t>0$, and there differentiation under the integral sign (Theorem 7.17(b)) gives the same derivative as the finite difference, to the precision of the quadrature. Both closed forms agree with numerical quadrature to about $10^{-8}$.